## Multi-Token Prediction (MTP)

### Step 0: Load packages

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

### Step 1: Define RMSNorm class

In [4]:
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps: float=1e-8):
        super().__init__()
        self.eps = eps
    def forward(self, x):
        # x:(batch, d_model)
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return x/rms

### Step 2: Define the Multi-Token Prediction class

In [ ]:
class SimpleMTP(nn.Module):
    def __init__(self, d_model, vocab_size: int, num_heads: int = 3, nhead: int = 2):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.num_heads = num_heads

        # shared modules
        self.rmsnorm = RMSNorm(d_model)
        self.embed = nn.Embedding(vocab_size, d_model)
        self.unembed = nn.Linear(d_model, vocab_size, bias=False)
        # share weights between embed and unembed
        self.unembed_weight = self.embed.weight

        # one projection + one transformer per head
        self.projections = nn.ModuleList([
            nn.Linear(2*d_model, d_model) for _ in range(num_heads)

        ])
        self.transformers = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead)
            for _ in range(num_heads)
        ])
        